# 전처리 파이프라인 — 학교명 댓글 데이터셋

In [24]:
import importlib
import preprocessing
importlib.reload(preprocessing)
from preprocessing import extract_nouns

JVMNotFoundException: No JVM shared library file (jvm.dll) found. Try setting up the JAVA_HOME environment variable properly.

In [ ]:
import sys
from pathlib import Path

# 커널이 어디서 시작되든(repo 루트든 src/ 안이든) 항상 동작하도록
# data/, src/ 폴더를 둘 다 가진 위치를 repo 루트로 잡는다.
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src"))

REPO_ROOT

WindowsPath('d:/Study/dongguk_university/dreampath')

In [ ]:
import pandas as pd

df = pd.read_csv(REPO_ROOT / "data" / "raw" / "dataset.csv", encoding="utf-8")
df.shape

(1000, 2)

## 전처리 1단계 — 이모지 / ㅋㅋㅎㅎ / 문장부호 제거

In [ ]:
from preprocessing import clean_text

df["comment_clean_1"] = df["comment"].apply(clean_text)
df[["comment", "comment_clean_1"]].head(20)

,comment,comment_clean_1
0,동국대학교,동국대학교
1,이번엔 중동고 차례입니다 🍗,이번엔 중동고 차례입니다
2,우리 반/동아리 대표로 서초초 신청합니다!,우리 반/동아리 대표로 서초초 신청합니다
3,잠실중 친구가 놀러왔지만 치킨은 건국대로 주세요,잠실중 친구가 놀러왔지만 치킨은 건국대로 주세요
4,이화여대부속초 친구랑 서초초 친구 같이 보고 있는데 저는 이화여대부속초로 신청!,이화여대부속초 친구랑 서초초 친구 같이 보고 있는데 저는 이화여대부속초로 신청
5,학식 말고 치킨 먹고 싶어요 인하대학교,학식 말고 치킨 먹고 싶어요 인하대학교
6,서울공업고.. 오늘만 기다렸어요,서울공업고 오늘만 기다렸어요
7,서강대 학생들 모여라,서강대 학생들 모여라
8,동아리방에서 기다릴게요 서초중,동아리방에서 기다릴게요 서초중
9,대치중 학생입니다 대치중 뽑아주세요,대치중 학생입니다 대치중 뽑아주세요


In [ ]:
# 실제로 뭔가 바뀐 행만 모아서 눈으로 확인
changed = df[df["comment"] != df["comment_clean_1"]]
print(f"{len(changed)}개 행에서 변경 발생")
changed[["comment", "comment_clean_1"]].head(30)

454개 행에서 변경 발생


,comment,comment_clean_1
1,이번엔 중동고 차례입니다 🍗,이번엔 중동고 차례입니다
2,우리 반/동아리 대표로 서초초 신청합니다!,우리 반/동아리 대표로 서초초 신청합니다
4,이화여대부속초 친구랑 서초초 친구 같이 보고 있는데 저는 이화여대부속초로 신청!,이화여대부속초 친구랑 서초초 친구 같이 보고 있는데 저는 이화여대부속초로 신청
6,서울공업고.. 오늘만 기다렸어요,서울공업고 오늘만 기다렸어요
14,잠원초등학교 학생입니다. 이벤트 참여합니다.,잠원초등학교 학생입니다 이벤트 참여합니다
16,잠실중 학생들 야근 말고 치킨 먹자 🙏,잠실중 학생들 야근 말고 치킨 먹자
25,배고픈 인하부초 학생들 살려주세요ㅠㅠ,배고픈 인하부초 학생들 살려주세요
31,한양대 응원단 여기 있습니다!!,한양대 응원단 여기 있습니다
34,중동고!!! 치킨 부탁드려요,중동고 치킨 부탁드려요
36,서초중!!! 치킨 부탁드려요,서초중 치킨 부탁드려요


In [ ]:
df.head(1)

,comment_id,comment,comment_clean_1
0,C0001,동국대학교,동국대학교


## 전처리 2단계 — 명사만 추출 (조사/동사 제거)

In [ ]:
from preprocessing import extract_nouns

df["comment_noun"] = df["comment_clean_1"].apply(extract_nouns)
df[["comment", "comment_noun"]].head(20)

ImportError: cannot import name 'extract_nouns' from 'preprocessing' (d:\Study\dongguk_university\dreampath\src\preprocessing.py)

In [ ]:
out_path = REPO_ROOT / "data" / "processed" / "preprocessing.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df[["comment_id", "comment", "comment_clean_1", "comment_noun"]].to_csv(
    out_path, index=False, encoding="utf-8-sig"
)
out_path